Step 0: Mounting Google Drive and creating the folders


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/doctalk'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/faiss_index', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/uploaded_docs', exist_ok=True)

print('Google Drive mounted and project folders ready.')
print(f'Project folder: {PROJECT_DIR}')

Step 1: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pymupdf
!pip install -q sentence-transformers

print('All dependencies installed.')

Step 2: Uploading Documents


In [ ]:
from google.colab import files
import shutil

print('Please upload a PDF file...')
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f'{PROJECT_DIR}/uploaded_docs/{filename}'
    shutil.copy(filename, dest)
    PDF_PATH = dest
    print(f'Saved to Drive: {dest}')

print(f'\n Working with: {PDF_PATH}')

Step 3: Loading and Extracting Text


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()

print(f'Loaded {len(pages)} pages from the PDF')
print(f'\n--- Preview of Page 1 (first 500 chars) ---')
print(pages[0].page_content[:500])
print(f'\n--- Metadata ---')
print(pages[0].metadata)

Step 4: Splitting the text into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len
)

chunks = splitter.split_documents(pages)

print(f'Split into {len(chunks)} chunks')
print(f'\n--- Example Chunk ---')
print(f'Content: {chunks[0].page_content}')
print(f'Metadata: {chunks[0].metadata}')
print(f'\n--- Next Chunk (notice the overlap) ---')
print(chunks[1].page_content[:200])

Step 5: Generate Embeddings with HuggingFace

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

print('Loading embedding model.')

embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Quick test — embed a single sentence
test_vector = embeddings.embed_query('This is a test sentence.')
print(f'\nEmbedding model loaded!')
print(f'Vector size: {len(test_vector)} dimensions')
print(f'First 5 values: {test_vector[:5]}')

Step 6: Store the embeddings in the vector store

In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm

print(f'Embedding {len(chunks)} chunks and building FAISS index...\n')

# Embed chunks in batches with a progress bar
batch_size = 50
all_embeddings = []

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks", unit="batch"):
    batch = chunks[i:i + batch_size]
    batch_texts = [chunk.page_content for chunk in batch]
    batch_embeddings = embeddings.embed_documents(batch_texts)
    all_embeddings.extend(batch_embeddings)

print(f'\nEmbedded {len(all_embeddings)} chunks!')

# Build FAISS index from embeddings
print('Building FAISS index...')
texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
vectorstore = FAISS.from_embeddings(
    list(zip(texts, all_embeddings)),
    embeddings,
    metadatas=metadatas
)

# Save to Drive
FAISS_PATH = f'{PROJECT_DIR}/faiss_index'
vectorstore.save_local(FAISS_PATH)

print(f'FAISS index saved to Drive!')
print(f'Location: {FAISS_PATH}')

Step 7: Query Test

In [ ]:
# Modify the query to something relevant to your document
query = ""  # enter your question here

results = vectorstore.similarity_search(query, k=5)

print(f'Query: "{query}"')
print(f'\nTop {len(results)} most relevant chunks:\n')

for i, doc in enumerate(results):
    print(f'--- Chunk {i+1} (Page {doc.metadata.get("page", 0) + 1}) ---')
    print(doc.page_content)
    print()